# Graph based analysis of interactions in Litigi community


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import networkx as nx
from functions.utilities import preprocess


## Load and preprocess data

In [2]:
filename='processed_data\itigi_comments_copnsolidated.parqet'
litigi=pd.read_parquet(filename).drop_duplicates(subset=['author','body','id', 'created_utc'])
litigi['created_dt']=(pd.to_datetime(litigi['created_utc'],unit='s'))
litigi['body_preprocessed']=litigi['body'].apply(lambda x: preprocess(x))

# extract the author of the parent comment (if applicable)
id_to_author = litigi.set_index('id')['author'].to_dict()
litigi['parent_author'] = litigi['parent_id'].str.split('_').str[1].map(id_to_author)

litigi.head()

,author,author_flair_css_class,author_flair_text,body,can_gild,collapsed,collapsed_reason,controversiality,created_utc,distinguished,...,top_awarded_type,comment_type,collapsed_reason_code,retrieved_utc,author_is_blocked,unrepliable_reason,editable,created_dt,body_preprocessed,parent_author
0,Trickyhere,None,None,Qualcosa da spaccare?,True,False,None,0,1502873647,None,...,None,None,None,NaN,None,NaN,None,2017-08-16 08:54:07,Qualcosa da spaccare?,NaN
1,panicClark,None,None,"Avevo quest'ideona dello sticky giornaliero ""A...",True,False,None,0,1502873820,None,...,None,None,None,NaN,None,NaN,None,2017-08-16 08:57:00,"Avevo quest'ideona dello sticky giornaliero ""A...",NaN
2,MeccAnon,None,None,"Risposta USENET: ""non l'avete capita"" (cit.)\n...",True,False,None,0,1502874353,None,...,None,None,None,NaN,None,NaN,None,2017-08-16 09:05:53,"Risposta USENET: ""non l'avete capita"" (cit.) ...",panicClark
3,panicClark,None,None,[GENIO.](https://www.reddit.com/r/litigi/comme...,True,False,None,0,1502876399,moderator,...,None,None,None,NaN,None,NaN,None,2017-08-16 09:39:59,GENIO.,MeccAnon
4,MeccAnon,None,None,&gt; nella sidebar trovate il suo manifesto or...,True,False,None,0,1502877513,None,...,None,None,None,NaN,None,NaN,None,2017-08-16 09:58:33,&gt; nella sidebar trovate il suo manifesto or...,NaN


## Extract subsample

In [3]:
litigi_subsample=litigi[:10000][['author','body_preprocessed','id', 'is_submitter', 'link_id', 'parent_id','created_dt','parent_author']]
litigi_subsample

,author,body_preprocessed,id,is_submitter,link_id,parent_id,created_dt,parent_author
0,Trickyhere,Qualcosa da spaccare?,dlp0rm3,False,t3_6u0lay,t3_6u0lay,2017-08-16 08:54:07,NaN
1,panicClark,"Avevo quest'ideona dello sticky giornaliero ""A...",dlp0tx1,False,t3_6u0lay,t3_6u0lay,2017-08-16 08:57:00,NaN
2,MeccAnon,"Risposta USENET: ""non l'avete capita"" (cit.) ...",dlp10tv,True,t3_6u0lay,t1_dlp0tx1,2017-08-16 09:05:53,panicClark
3,panicClark,GENIO.,dlp1ryh,False,t3_6u0lay,t1_dlp10tv,2017-08-16 09:39:59,MeccAnon
4,MeccAnon,&gt; nella sidebar trovate il suo manifesto or...,dlp26uj,False,t3_6twu0x,t3_6twu0x,2017-08-16 09:58:33,NaN
...,...,...,...,...,...,...,...,...
10024,r0seMvp,l ho cancellato io,dvylm1d,False,t3_85lxj2,t1_dvydeta,2018-03-19 21:07:57,panicClark
10025,DanziRevenge,"È il maggior produttore di stronzate, si prend...",dvzex3m,False,t3_85l57e,t1_dvy9zs1,2018-03-20 07:05:12,[deleted]
10026,DanziRevenge,È meglio morire a 2 mesi dal concepimento o na...,dvzf63j,False,t3_85m7dm,t3_85m7dm,2018-03-20 07:15:22,NaN
10027,DanziRevenge,Ma che sfiga ha questo? Praticamente gli han r...,dvzfr2l,False,t3_85lxj2,t3_85lxj2,2018-03-20 07:39:35,NaN


## Extract user interactions

In [4]:
user_interactions=litigi_subsample.groupby(['author','parent_author']).agg(**{'count':('id','count')}).reset_index()
user_interactions

,author,parent_author,count
0,-_Numb_-,-_Numb_-,1
1,-_Numb_-,SpiegoLeDiscussioni,5
2,-_Numb_-,[deleted],1
3,-_Numb_-,panicClark,1
4,00112321100,SpiegoLeDiscussioni,1
...,...,...,...
1563,xImReD,Trickyhere,2
1564,xImReD,panicClark,1
1565,youtubefactsbot,[deleted],1
1566,zakk,UnaMammaPreoccupata,1


In [6]:
G=nx.DiGraph()
G.add_edges_from([a for a in zip(user_interactions['author'],user_interactions['parent_author'],[{'log_weight': np.log(c+1),'weight': c} for c in user_interactions['count'] ])])


In [7]:
from functions.analysis import symmetryze_graph
G_sym=symmetryze_graph(G)
nx.node_link_data(G_sym)

{'directed': False,
 'multigraph': False,
 'graph': {},
 'nodes': [{'id': '-_Numb_-'}],
 'links': [{'weight': 2, 'source': '-_Numb_-', 'target': '-_Numb_-'}]}

## Export and visualize network

In [ ]:
from pyvis.network import Network


options = {
  "physics": {
    "enabled": False,
    "barnesHut": {
      "centralGravity": 1.6,
      "springLength": 140,
      "springConstant": 0.1
    },
    "minVelocity": 0.75,
    "timestep": 0.11
  }
}

# Convert NetworkX graph to Pyvis
net = Network(height="750px", width="100%", bgcolor="#222222", font_color="white", select_menu=True)
net.show_buttons()
net.from_nx(G)
net.toggle_physics(False)
net.save_graph("litigi.html")

## Sources

https://towardsdatascience.com/large-graph-visualization-tools-and-approaches-2b8758a1cd59?gi=130d1c56065b

For a chord diagram look into: https://plotly.com/python/v3/chord-diagram/